![](../assets/placeholder.png)

## AI Ajanları Oluşturmaya Giriş
Bu notebook'ta, Langchain kullanarak bir AI ajanı oluşturma yolculuğuna başlıyoruz. Bu kapsamlı rehber, önceki derslerden elde edilen bilginin sentezini yaparak, görevleri bağımsız olarak gerçekleştirebilecek bir ajan oluşturmaya yönelik adım adım bir yaklaşım sağlar.

In [ ]:
from dotenv import load_dotenv
import os

# .env dosyasından ortam değişkenlerini yükleyin
load_dotenv()

#### Ajanlar için Özel Araçların Ayarlanması
Ajanımızın yeteneklerini geliştirmek için, önceki notebook'ta tanımlanan özel araçları entegre edeceğiz. Bu araçlar (bir URL içerik getiricisi ve bir özeti çıkaran dahil) ajanımıza bilgiyi verimli bir şekilde almak ve işlemek için güç verecektir.

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain.tools import tool

@tool
def fetch_url_content(url: str) -> str:
    """Belirtilen bir URL'nin içeriğini alır. Bu araç, web tabanlı bilgilere ihtiyaç duyan ajanlar için gereklidir."""
    loader = WebBaseLoader(url)
    docs = loader.load()
    return docs

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_anthropic import ChatAnthropic

@tool
def summarize_content(content: str) -> str:
    """Metin tabanlı içeriğinin kısa bir özetini üretir. Özet görevleri için optimize edilmiş bu araç, temel bilgileri damıtlamaya yardımcı olur."""
    system_message = "Lütfen içerik için kısa bir özet sağlayın"
    human_message = "İçerik: " + str(content)
    
    messages = [
        SystemMessage(content=system_message),
        HumanMessage(content=human_message),
    ]
    model = ChatAnthropic(model="claude-3-haiku-20240229")
    parser = StrOutputParser()
    
    chain = model | parser
    summary = chain.invoke(messages)
    return summary

In [ ]:
# Özel araçları ajanla entegre edin
tools = [fetch_url_content, summarize_content]

In [ ]:
# Ajanın çekirdeği için LLM modelini seçin
from langchain_openai import ChatOpenAI
model = ChatOpenAI(model="gpt-4o")

#### Langgraph ile Ajanın İnşa Edilmesi
Ajanımızı oluşturmak için Langgraph'ın ReAct ajan çerçevesini kullanacağız. Bu modern yaklaşım, araçlar ve modellerin sorunsuz şekilde entegrasyonunu sağlayarak çok yönlü ve özerk bir ajan oluşturmayı kolaylaştırır.

In [ ]:
from langgraph.prebuilt import create_react_agent

# Seçilen model ve araçlarla ajan yürütücüsünü oluşturun
agent_executor = create_react_agent(model, tools)

In [ ]:
# Ajanın ilk yanıtını test edin
agent_executor.invoke({"messages": [HumanMessage(content="merhaba!")]})

In [ ]:
# Ajantan web içeriğini özetlemesini isteyin
# Not: Yürütme süresi, içerik boyutu nedeniyle LLM'nin jeton işlem sınırına bağlı olarak değişebilir.
response = agent_executor.invoke({"messages": [HumanMessage(content="lütfen bu url'nin içeriğini özetleyebilir misiniz https://lilianweng.github.io/posts/2023-06-23-agent/")]})
response["messages"]